# Feature Engineering — Port Calls Dataset

This notebook loads `data/port_calls.parquet`, runs `build_features` from `features.py`,
and explores the resulting feature set through EDA, correlation analysis, and causal feature inspection.

**Dataset:** 76,000 port call records · Jan 2024 – Dec 2025 · Haifa & Ashdod

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Imports OK')

## 1. Load Raw Data

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'port_calls.parquet')

df_raw = pd.read_parquet(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

## 2. Build Features

In [ ]:
from features import build_features, ALL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_ENC_FEATURES

print('Running build_features() — rolling windows may take ~1-2 min ...')
df = build_features(df_raw)

print(f'\nFeature shape:  {df.shape}')
print(f'Feature count:  {len(ALL_FEATURES)} (ALL_FEATURES list)')
print(f'  Numeric:      {len(NUMERIC_FEATURES)}')
print(f'  Categorical:  {len(CATEGORICAL_ENC_FEATURES)}')
print(f'\nNew columns added: {df.shape[1] - df_raw.shape[1]}')

## 3. EDA

### 3a. Distribution of `waiting_anchor_hours`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full distribution
ax = axes[0]
ax.hist(df['waiting_anchor_hours'], bins=80, color='steelblue', edgecolor='none', alpha=0.85)
ax.set_xlabel('Waiting time at anchor (hours)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Anchorage Waiting Time')
for p, label in [(80, 'P80'), (95, 'P95'), (99, 'P99')]:
    v = np.percentile(df['waiting_anchor_hours'], p)
    ax.axvline(v, linestyle='--', linewidth=1.2, label=f'{label}={v:.1f}h')
ax.legend(fontsize=9)

# Log-scale inset for heavy tail
ax2 = axes[1]
ax2.hist(df['waiting_anchor_hours'].clip(0, 96), bins=80,
         color='darkorange', edgecolor='none', alpha=0.85)
ax2.set_yscale('log')
ax2.set_xlabel('Waiting time (hours, clipped at 96h)')
ax2.set_ylabel('Count (log scale)')
ax2.set_title('Waiting Time — Log Scale (clipped to 96h)')

plt.tight_layout()
plt.show()

print('\nPercentile summary:')
for p in [50, 75, 80, 90, 95, 99]:
    v = np.percentile(df['waiting_anchor_hours'], p)
    print(f'  P{p:>2}: {v:.2f}h')
print(f'  Mean: {df["waiting_anchor_hours"].mean():.2f}h')
print(f'  % direct berth (0h): {(df["waiting_anchor_hours"] == 0).mean():.1%}')

### 3b. Vessel Type Counts by Port

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, port in zip(axes, ['Haifa', 'Ashdod']):
    counts = (
        df[df['port_name'] == port]['vessel_type']
        .value_counts()
        .sort_values(ascending=True)
    )
    colors = plt.cm.tab10.colors[:len(counts)]
    bars = ax.barh(counts.index, counts.values, color=colors, edgecolor='none')
    ax.set_xlabel('Number of port calls')
    ax.set_title(f'Vessel Types — {port} ({counts.sum():,} calls)')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('\nVessel type share across both ports:')
print(df['vessel_type'].value_counts(normalize=True).mul(100).round(1).to_string())

### 3c. Daily Port Call Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Daily call volume over time
ax = axes[0, 0]
daily = df.set_index('ata_actual').resample('D')['id'].count()
daily.plot(ax=ax, color='steelblue', linewidth=0.7, alpha=0.8)
daily.rolling(14).mean().plot(ax=ax, color='red', linewidth=1.5, label='14-day MA')
ax.set_xlabel('')
ax.set_ylabel('Port calls per day')
ax.set_title('Daily Port Call Volume (Jan 2024 – Dec 2025)')
ax.legend(fontsize=9)

# Hour of day
ax = axes[0, 1]
df.groupby('hour_of_day')['id'].count().plot(kind='bar', ax=ax,
    color='steelblue', edgecolor='none')
ax.set_xlabel('Hour of day (ATA)')
ax.set_ylabel('Count')
ax.set_title('Arrivals by Hour of Day')
ax.tick_params(axis='x', rotation=0)

# Day of week
ax = axes[1, 0]
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_counts = df.groupby('day_of_week')['id'].count()
ax.bar(dow_labels, dow_counts.values, color='darkorange', edgecolor='none')
ax.set_xlabel('Day of week')
ax.set_ylabel('Count')
ax.set_title('Arrivals by Day of Week')

# Month
ax = axes[1, 1]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df.groupby('month')['id'].count()
ax.bar(month_labels[:len(month_counts)], month_counts.values,
       color='seagreen', edgecolor='none')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
ax.set_title('Arrivals by Month')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Correlation Heatmap — Top Features vs `waiting_anchor_hours`

In [ ]:
TARGET = 'waiting_anchor_hours'

# Select numeric features present in df
feat_cols = [c for c in ALL_FEATURES if c in df.columns]
corr_with_target = (
    df[feat_cols + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=False)
)

top_feats = corr_with_target.head(20).index.tolist()

corr_matrix = df[top_feats + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)
# No masking — show full square for readability
sns.heatmap(
    corr_matrix,
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7},
    linewidths=0.3,
    square=True,
    cbar_kws={'shrink': 0.7},
)
ax.set_title('Pearson Correlation — Top 20 Features + Target', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 10 features by |correlation| with waiting_anchor_hours:')
for rank, (feat, val) in enumerate(corr_with_target.head(10).items(), 1):
    raw_corr = df[feat_cols + [TARGET]].corr()[TARGET].drop(TARGET)[feat]
    print(f'  {rank:>2}. {feat:<35}  r = {raw_corr:+.4f}')

## 5. Causal Feature Importance — `berth_competition` & `weather_wind_knots`

These two columns are the **generative causal inputs** used in the simulation to produce
`waiting_anchor_hours`. Plotting them confirms the causal signal is captured by the features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- berth_competition vs waiting (scatter) ---
ax = axes[0]
sample = df.sample(min(5000, len(df)), random_state=42)
ax.scatter(sample['berth_competition'], sample['waiting_anchor_hours'],
           alpha=0.15, s=6, color='steelblue')
# Bin mean overlay
bins = np.arange(0, sample['berth_competition'].max() + 0.25, 0.25)
sample['bc_bin'] = pd.cut(sample['berth_competition'], bins=bins)
bin_means = sample.groupby('bc_bin', observed=True)['waiting_anchor_hours'].mean()
bin_centers = bins[:-1] + 0.125
ax.plot(bin_centers[:len(bin_means)], bin_means.values, color='red',
        linewidth=2, label='Bin mean')
ax.set_xlabel('berth_competition')
ax.set_ylabel('waiting_anchor_hours')
ax.set_title('Berth Competition vs Waiting Time')
ax.legend(fontsize=9)

# --- weather_wind_knots vs waiting (scatter) ---
ax = axes[1]
ax.scatter(sample['weather_wind_knots'], sample['waiting_anchor_hours'],
           alpha=0.15, s=6, color='darkorange')
bins_w = np.arange(0, sample['weather_wind_knots'].max() + 2, 2)
sample['wind_bin'] = pd.cut(sample['weather_wind_knots'], bins=bins_w)
bin_means_w = sample.groupby('wind_bin', observed=True)['waiting_anchor_hours'].mean()
bin_centers_w = bins_w[:-1] + 1
ax.plot(bin_centers_w[:len(bin_means_w)], bin_means_w.values, color='red',
        linewidth=2, label='Bin mean')
ax.set_xlabel('weather_wind_knots')
ax.set_ylabel('waiting_anchor_hours')
ax.set_title('Wind Speed vs Waiting Time')
ax.legend(fontsize=9)

# --- Distributions of both causal features ---
ax = axes[2]
ax.hist(df['weather_wind_knots'], bins=50, alpha=0.6,
        color='darkorange', label='wind_knots', density=True)
ax2r = ax.twinx()
ax2r.hist(df['berth_competition'], bins=30, alpha=0.5,
          color='steelblue', label='berth_competition', density=True)
ax.set_xlabel('Value')
ax.set_ylabel('Density (wind)', color='darkorange')
ax2r.set_ylabel('Density (competition)', color='steelblue')
ax.set_title('Causal Feature Distributions')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.show()

r_bc   = df[['berth_competition', 'waiting_anchor_hours']].corr().iloc[0, 1]
r_wind = df[['weather_wind_knots', 'waiting_anchor_hours']].corr().iloc[0, 1]
print(f'berth_competition  → waiting_anchor_hours  r = {r_bc:+.4f}')
print(f'weather_wind_knots → waiting_anchor_hours  r = {r_wind:+.4f}')
print(f'weather_storm_flag → waiting (storm ≥25kn mean): '
      f'{df[df["weather_storm_flag"]==1]["waiting_anchor_hours"].mean():.2f}h '
      f'vs non-storm: {df[df["weather_storm_flag"]==0]["waiting_anchor_hours"].mean():.2f}h')

## 6. Feature Summary

In [ ]:
feat_present = [c for c in ALL_FEATURES if c in df.columns]
feat_missing = [c for c in ALL_FEATURES if c not in df.columns]

print('=== Feature Engineering Summary ===')
print(f'Raw columns:            {df_raw.shape[1]}')
print(f'Engineered columns:     {df.shape[1]}')
print(f'ALL_FEATURES list size: {len(ALL_FEATURES)}')
print(f'  Present in df:        {len(feat_present)}')
print(f'  Missing from df:      {len(feat_missing)}')
if feat_missing:
    print(f'  Missing: {feat_missing}')

print(f'\nTotal rows:     {len(df):,}')
print(f'Date range:     {df["ata_actual"].min().date()} → {df["ata_actual"].max().date()}')
print(f'Ports:          {dict(df["port_name"].value_counts())}')

print('\nNaN counts in feature columns:')
nan_counts = df[feat_present].isna().sum()
nan_nonzero = nan_counts[nan_counts > 0]
if len(nan_nonzero) == 0:
    print('  None — all features complete.')
else:
    print(nan_nonzero.to_string())

print('\nSample feature dtypes:')
print(df[feat_present[:10]].dtypes.to_string())